<!--nav--> [🗺 Learning path](README.md) · **25/46** · ◀ [vLLM High-Throughput Serving](./vLLM_High_Throughput_Serving.ipynb) · [Speculative Decoding](./Speculative_Decoding_Advanced_Serving.ipynb) ▶

# Quantized Serving Showdown: FP16 vs AWQ vs GPTQ (vs NF4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugeerth/gpu-training-notebooks/blob/main/Quantized_Serving_Showdown.ipynb)

[Serving Fundamentals](./Serving_Fundamentals_KV_Cache_Batching.ipynb) proved decode is **memory-bandwidth
bound**: every token requires reading every weight from HBM. Quantization attacks that directly —
**int4 weights are 4× fewer bytes to read than fp16**, so decode gets faster *and* the freed memory
becomes KV cache (= more concurrent users). It's the rare optimization that improves latency,
throughput, and capacity at once... in exchange for some accuracy. Today we measure all four sides
of that trade.

**The showdown:** one model — `Qwen2.5-1.5B-Instruct` — in four formats, each benchmarked identically:

| Format | Bits | How it was made | Served by |
|---|---|---|---|
| **FP16** | 16 | the original | vLLM |
| **AWQ** | 4 (group=128) | activation-aware offline quantization | vLLM (int4 kernels) |
| **GPTQ** | 4 (group=128) | second-order offline quantization | vLLM (int4 kernels) |
| **NF4** | 4 | bitsandbytes on-the-fly at load | transformers |

For each: **weight memory · batch throughput · single-user decode speed · accuracy** on a 20-question quiz.

**Runs on:** free Colab **T4** · GPU required. Everything also works on A100/L4 (faster, and unlocks
Marlin int4 kernels).

## The 5-minute theory: how do you delete 12 of every 16 bits?

**Weight-only int4** stores weights as 4-bit integers plus, for every *group* of 128 weights, one
fp16 `scale` (and often a `zero` point): `w ≈ scale × (q - zero)`. At inference the kernel
dequantizes on the fly — the GPU reads 4-bit weights (the slow part, now 4× smaller) and does the
math in fp16 (the fast part, unchanged). That's why weight-only quantization speeds up *decode*
(bandwidth-bound) but barely helps *prefill* (compute-bound).

Naively rounding every weight to 4 bits noticeably hurts quality. The two dominant fixes:

- **GPTQ** (2022): quantize one column at a time, then update not-yet-quantized weights to
  *compensate* for the error just introduced, using second-order (Hessian) information from a small
  calibration set. Error doesn't accumulate — it gets actively cancelled.
- **AWQ** (2023): observation — ~1% of weight channels are "salient" because they meet **large
  activations**. AWQ finds a per-channel scaling that shrinks activations where weights matter most
  (and grows the weights to compensate), *then* rounds. No Hessian, more robust off-distribution;
  it's become the default choice for serving.

Both are **offline, one-time** processes (minutes on one GPU, via
[llm-compressor](https://github.com/vllm-project/llm-compressor) or AutoAWQ/GPTQModel) producing a
checkpoint you download — Qwen publishes official AWQ and GPTQ variants of every size, which is
what we use.

**NF4 (bitsandbytes)** is different: it quantizes *at load time* with no calibration. That's exactly
what QLoRA uses for **training** ([LoRA & QLoRA Fine-Tuning](./LoRA_QLoRA_FineTuning.ipynb)). We include it to show why "the QLoRA trick" is not a
*serving* strategy — its dequant path is built for flexibility, not decode speed.

**FP8 (E4M3)** is the datacenter option: ~fp16 quality at half the bytes, but needs Ada/Hopper/
Blackwell tensor cores (T4 is Turing, so we explain it and move on — the code is one flag:
`--quantization fp8`).

In [ ]:
!pip install -q -U vllm openai bitsandbytes transformers accelerate

import torch, json, subprocess, sys, time, os
assert torch.cuda.is_available(), "GPU required - Runtime > Change runtime type > T4"
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} · {p.total_memory/1e9:.1f} GB · SM {p.major}.{p.minor}")

## The benchmark harness

One subtlety worth stealing for your own benchmarks: loading three vLLM engines back-to-back in one
Python process is flaky (CUDA memory isn't reliably returned until the process exits). So the
harness below runs **each engine in a fresh subprocess** and reports JSON back — every format gets
an identical, clean GPU. Each run measures:

1. **weight memory** (allocated right after load),
2. **batch throughput** — 32 prompts × 128 tokens, continuous batching,
3. **interactive decode** — 1 request, tokens/s (the streaming speed a single user feels),
4. **accuracy** — 20 exact-answer questions (capitals, arithmetic, science), greedy decoding.

20 questions is a smoke test, not an eval — it catches "quantization broke the model", while real
decisions deserve `lm-eval` on MMLU/GSM8K. We say this out loud so nobody ships on a 20-question quiz.

In [ ]:
BENCH = r'''
import json, time, sys, torch
MODEL = sys.argv[1]

QA = [("What is the capital of France?", "paris"),
      ("What is 17 + 25?", "42"),
      ("What is the chemical symbol for gold?", "au"),
      ("How many legs does a spider have?", "8"),
      ("What planet is known as the Red Planet?", "mars"),
      ("What is 12 times 12?", "144"),
      ("What is the capital of Japan?", "tokyo"),
      ("What gas do plants absorb from the atmosphere?", "carbon dioxide"),
      ("What is the largest ocean on Earth?", "pacific"),
      ("What is 100 divided by 4?", "25"),
      ("Who wrote Romeo and Juliet?", "shakespeare"),
      ("What is the boiling point of water in Celsius?", "100"),
      ("What is the capital of Australia?", "canberra"),
      ("How many sides does a hexagon have?", "6"),
      ("What is the square root of 81?", "9"),
      ("What metal is liquid at room temperature?", "mercury"),
      ("What is the capital of Canada?", "ottawa"),
      ("How many minutes are in three hours?", "180"),
      ("What is the hardest natural substance?", "diamond"),
      ("What is 15 percent of 200?", "30")]

from vllm import LLM, SamplingParams
llm = LLM(model=MODEL, dtype="half", max_model_len=2048, gpu_memory_utilization=0.85)
weight_gb = torch.cuda.memory_allocated() / 1e9   # dominated by weights (KV pool grows separately)

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL)
def chat(u): return tok.apply_chat_template([{"role":"user","content":u}],
                                            add_generation_prompt=True, tokenize=False)

# 2) batch throughput
prompts = [chat(f"Describe invention number {i} that changed daily life, in about 80 words.")
           for i in range(32)]
sp = SamplingParams(temperature=0.8, top_p=0.95, max_tokens=128)
llm.generate([chat("warmup")], SamplingParams(max_tokens=8))
t0 = time.perf_counter(); outs = llm.generate(prompts, sp); dt = time.perf_counter() - t0
batch_tps = sum(len(o.outputs[0].token_ids) for o in outs) / dt

# 3) interactive single-stream decode
sp1 = SamplingParams(temperature=0.0, max_tokens=128, ignore_eos=True)
t0 = time.perf_counter(); out = llm.generate([chat("Tell me a long story about the ocean.")], sp1)
single_tps = len(out[0].outputs[0].token_ids) / (time.perf_counter() - t0)

# 4) accuracy
qs = [chat(q + " Answer with just the answer, nothing else.") for q, _ in QA]
outs = llm.generate(qs, SamplingParams(temperature=0.0, max_tokens=16))
correct = sum(a in o.outputs[0].text.lower() for o, (_, a) in zip(outs, QA))

print("RESULT " + json.dumps(dict(model=MODEL, weight_gb=round(weight_gb, 2),
      batch_tps=round(batch_tps, 1), single_tps=round(single_tps, 1),
      accuracy=f"{correct}/{len(QA)}")))
'''

with open("bench_one.py", "w") as f:
    f.write(BENCH)

def bench(model_id):
    print(f"benchmarking {model_id} (fresh subprocess, few minutes)...")
    r = subprocess.run([sys.executable, "bench_one.py", model_id],
                       capture_output=True, text=True, timeout=1800,
                       env={**os.environ, "VLLM_LOGGING_LEVEL": "WARNING"})
    for line in r.stdout.splitlines():
        if line.startswith("RESULT "):
            return json.loads(line[7:])
    raise RuntimeError(f"bench failed for {model_id}:\n{r.stdout[-2000:]}\n{r.stderr[-2000:]}")

In [ ]:
# The showdown. ~10-15 min total on a T4 (each subprocess pays engine startup once).
results = [bench(m) for m in [
    "Qwen/Qwen2.5-1.5B-Instruct",             # FP16 reference
    "Qwen/Qwen2.5-1.5B-Instruct-AWQ",         # official AWQ int4, group=128
    "Qwen/Qwen2.5-1.5B-Instruct-GPTQ-Int4",   # official GPTQ int4, group=128
]]

fp16 = results[0]
print(f"\n{'format':<12}{'weights':>9}{'batch tok/s':>13}{'1-user tok/s':>14}{'quiz':>7}")
print("-" * 55)
for r in results:
    name = ("FP16" if "AWQ" not in r["model"] and "GPTQ" not in r["model"]
            else "AWQ-int4" if "AWQ" in r["model"] else "GPTQ-int4")
    print(f"{name:<12}{r['weight_gb']:>7.2f}GB{r['batch_tps']:>13.0f}{r['single_tps']:>14.0f}{r['accuracy']:>7}")
print(f"\nvs FP16: AWQ weights {results[1]['weight_gb']/fp16['weight_gb']:.0%}, "
      f"single-user speed {results[1]['single_tps']/fp16['single_tps']:.2f}x")

### How to read the table

- **Weights: ~4× smaller.** On this 1.5B toy that's ~2 GB freed; on a 70B it's the difference
  between "needs 4×A100" and "fits on one". The freed VRAM becomes **KV cache** → more concurrent
  sequences → higher ceilings on exactly the continuous-batching throughput from [vLLM High-Throughput Serving](./vLLM_High_Throughput_Serving.ipynb).
- **Single-user decode: faster** — the weight-read per token shrank 4×. (On a T4 the speedup is real
  but below 4×: dequant costs compute, and Turing lacks the fused **Marlin** kernels that get int4
  close to its bandwidth-limit speedup on Ampere+.)
- **Batch throughput: smaller gain than single-user.** As batch size grows, decode shifts from
  bandwidth-bound toward compute-bound, and dequant overhead eats in. Quantization shines brightest
  at *interactive* batch sizes — which is what most latency-sensitive products run at.
- **Quiz: expect FP16-parity.** 4-bit AWQ/GPTQ typically costs ~0.1–0.5 perplexity points and a
  couple of MMLU points on well-calibrated checkpoints — real, but usually invisible on a smoke
  test. If your numbers dropped sharply, that's a red flag for that particular checkpoint.

## And NF4? Why the QLoRA trick isn't a serving format

bitsandbytes NF4 quantizes at load time (no calibration) — perfect for *training* on a budget
(QLoRA freezes NF4 weights and trains LoRA adapters on top). But watch what happens to decode speed:

In [ ]:
# NF4 on-the-fly quantization via transformers - measure the same single-stream decode.
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import gc

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
text = tok.apply_chat_template([{"role":"user","content":"Tell me a long story about the ocean."}],
                               add_generation_prompt=True, tokenize=False)
msgs = tok(text, return_tensors="pt").input_ids

def hf_decode_tps(quant_cfg, label):
    model = AutoModelForCausalLM.from_pretrained(
        MODEL, dtype=torch.float16, quantization_config=quant_cfg,
        device_map="cuda").eval()
    mem = torch.cuda.memory_allocated() / 1e9
    ids = msgs.to("cuda")
    with torch.no_grad():
        model.generate(ids, max_new_tokens=16, do_sample=False, pad_token_id=tok.eos_token_id)
        torch.cuda.synchronize(); t0 = time.perf_counter()
        model.generate(ids, max_new_tokens=128, min_new_tokens=128, do_sample=False,
                       pad_token_id=tok.eos_token_id)
        torch.cuda.synchronize(); tps = 128 / (time.perf_counter() - t0)
    print(f"{label:<28}{mem:>6.2f} GB weights {tps:>8.0f} tok/s")
    del model; gc.collect(); torch.cuda.empty_cache()
    return tps

fp16_tps = hf_decode_tps(None, "transformers FP16")
nf4_tps = hf_decode_tps(BitsAndBytesConfig(load_in_4bit=True,
                                           bnb_4bit_quant_type="nf4",
                                           bnb_4bit_compute_dtype=torch.float16),
                        "transformers NF4 (bnb)")
print(f"\nNF4 memory: great. NF4 decode speed: {nf4_tps/fp16_tps:.2f}x fp16 - "
      "often SLOWER, its dequant path isn't built for serving.")
print("Lesson: quantization format and inference kernels are a package deal.")

## The bigger map: which format when

| Situation | Reach for | Why |
|---|---|---|
| Serving on Ampere/older, or common cloud GPUs | **AWQ int4** (or GPTQ) | mature vLLM kernels (Marlin on SM80+), official checkpoints everywhere |
| H100/H200/Blackwell fleet | **FP8** (`--quantization fp8`, E4M3) | ~lossless, tensor-core native, also halves KV with `--kv-cache-dtype fp8` |
| Extreme squeeze (edge, huge models) | int4 AWQ + **FP8/int4 KV cache** | KV becomes the next bottleneck once weights shrink |
| Fine-tuning cheaply | **NF4 + QLoRA** ([LoRA & QLoRA Fine-Tuning](./LoRA_QLoRA_FineTuning.ipynb)) | that's what it's for — then *re-quantize properly* (AWQ/FP8) to serve |
| Making your own quant of a fine-tune | [llm-compressor](https://github.com/vllm-project/llm-compressor) | one script: calibrate → quantize → save a vLLM-ready checkpoint |

Two forward pointers, both about *the cache* rather than the weights:
- **KV cache quantization** — vLLM's `--kv-cache-dtype fp8` halves KV memory (Ada/Hopper+), doubling
  how many concurrent conversations fit. The KV formula from [Serving Fundamentals](./Serving_Fundamentals_KV_Cache_Batching.ipynb) is why this matters so much.
- **Quantization + speculation stack**: a quantized target model verifying a tiny draft model —
  next notebook.

## Recap

- Weight-only int4 (AWQ/GPTQ) ≈ **4× less weight memory**, faster interactive decode, ~free accuracy
  on good checkpoints — measured, not asserted.
- The mechanism is bandwidth: decode reads bytes, quantization shrinks bytes.
- Kernels decide everything: the same 4 bits are fast in vLLM's AWQ path and slow in bitsandbytes,
  because one was built for serving and the other for training.
- Smoke-test quality yourself, then confirm with `lm-eval` before shipping.

### Further reading
- [AWQ paper](https://arxiv.org/abs/2306.00978) · [GPTQ paper](https://arxiv.org/abs/2210.17323) · [FP8 formats (E4M3/E5M2)](https://arxiv.org/abs/2209.05433)
- [vLLM quantization docs](https://docs.vllm.ai/en/latest/features/quantization/) · [llm-compressor](https://github.com/vllm-project/llm-compressor)
- [Marlin int4 kernels](https://github.com/IST-DASLab/marlin) — how int4 actually hits its speedup on Ampere+

▶ **Next:** [Speculative Decoding & the Serving Frontier](./Speculative_Decoding_Advanced_Serving.ipynb) —
spend the GPU's idle compute on guessing the future.